<a href="https://colab.research.google.com/github/zo-ff/pandas/blob/main/asfreq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**■ asfreq() でよく使う引数**

時系列データの周期（頻度）を変更する際に使います。

* `freq` （必須）: `'D'`（日次）, `'B'`（営業日次）, `'H'`（時間次）, `'MS'`（月初）など
* `fill_value` : 新たに発生した欠損値を一律で特定の値で埋めたい場合に指定（例: `fill_value=0`）
* `method` : 簡易的な穴埋めを同時に行う場合に指定（例: `'ffill'`=前値保持, `'bfill'`=後値保持）
  ※ `interpolate()` で補間する場合は、ここでは `method` を指定せず NaN のまま次に渡します。

---

**■ interpolate() でよく使う引数**

`asfreq()` で生じた欠損値（NaN）を補間する際に使います。

* `method` : 補間アルゴリズムを指定（デフォルトは `'linear'` = 線形補間）
  * `'linear'`: 前後の値の間を直線でつなぐ（高速・シンプル）
  * `'time'`: 日付の間隔を考慮して線形補間する（不規則なデータに有効）
  * `'quadratic'` / `'cubic'`: スプライン補間（滑らかな曲線にしたい場合）
* `limit` : 連続して補間するNaNの最大個数を指定（例: `limit=2`）
* `limit_direction` : 補間の方向（`'forward'`, `'backward'`, `'both'`）。リーク防止には `'forward'`
* `limit_area` : 補間する領域を制限（`'inside'`: 内側のみ, `'outside'`: 外側のみ）

---

**■ ffill / bfill と interpolate() の使い分けの基準**

① `ffill`（または `asfreq(method='ffill')`）だけでいいケース
* 前回の値を「次のデータが来るまでそのまま維持する」のが自然なデータ
* 例: **設定値、契約ステータス、口座残高、マスタデータ**など（階段状に変化する）

② `interpolate()` を使うべきケース
* データとデータの間が「なだらかに変化している」と仮定するのが自然なデータ
* 例: **気温、センサーの測定値、売上の推移、株価**など

In [1]:
import pandas as pd
import numpy as np

# 1. サンプルデータの作成 (不規則な日付の売上データ)
dates = pd.to_datetime(['2026-06-01', '2026-06-03', '2026-06-06'])
df = pd.DataFrame({'sales': [100, 120, 150]}, index=dates)
print("--- 元データ ---")
print(df)

# 2. 前値で埋めるだけの場合 (interpolateは使わない)
print("\n--- ffillのみ適用 (前値保持) ---")
df_ffill = df.asfreq('D').ffill()
print(df_ffill)

# 3. 時間ベースで線形補間する場合 (最大連続2日まで)
print("\n--- asfreq + interpolate (時間ベース線形補間) ---")
df_interpolated = (
    df.asfreq('D')
      .interpolate(method='time', limit=2, limit_direction='forward')
)
print(df_interpolated)

--- 元データ ---
            sales
2026-06-01    100
2026-06-03    120
2026-06-06    150

--- ffillのみ適用 (前値保持) ---
            sales
2026-06-01  100.0
2026-06-02  100.0
2026-06-03  120.0
2026-06-04  120.0
2026-06-05  120.0
2026-06-06  150.0

--- asfreq + interpolate (時間ベース線形補間) ---
            sales
2026-06-01  100.0
2026-06-02  110.0
2026-06-03  120.0
2026-06-04  130.0
2026-06-05  140.0
2026-06-06  150.0
